
# Notebook 17 v2 — Residual Topology Geometry

This version refines Notebook 17 for paper-ready outputs.

Main upgrades:

- self-contained load/regenerate behavior,
- smoother residual field diagnostics,
- curvature renamed to **residual bend energy**,
- smoothed bend energy replaces noisy raw second derivatives,
- residual geometry map promoted as hero figure,
- FFT kept as secondary diagnostic,
- cleaner summary tables.

Core claim:

```text
The shared transition persists; topology reappears as structured residual geometry.
```


## Imports and setup

In [ ]:

import json
import zipfile
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import savgol_filter

np.random.seed(42)

FIG_DIR = Path("figures")
RESULTS_DIR = Path("results")
DOCS_DIR = Path("docs")

FIG_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
DOCS_DIR.mkdir(exist_ok=True)

GRAPH_SIZES = [16, 32, 64, 128]
TOPOLOGIES = [
    "ring_lattice",
    "small_world",
    "erdos_renyi",
    "scale_free",
    "modular_clustered",
]

NOISE_GRID = np.linspace(0.0, 0.40, 81)
MIDPOINT_LEVEL = 0.50
EPSILON = 0.05

print("cwd:", os.getcwd())
print("Ready.")


## Shared profile helpers

In [ ]:

def logistic_z(z):
    return 1 / (1 + np.exp(-z))

def shared_profile(z):
    # Decreasing bounded transition profile.
    return 1 - logistic_z(z)

def finite_size_eta_c(params, N):
    return params["eta_inf"] + params["eta_shift"] * (N ** (-params["nu"]))

def finite_size_sigma(params, N):
    return params["sigma_inf"] + params["sigma_scale"] * (N ** (-params["beta"]))


## Load existing Notebook 16 outputs or regenerate compatible data

In [ ]:

required_files = {
    "collapse": RESULTS_DIR / "renormalized_collapse_data.csv",
    "windows": RESULTS_DIR / "universality_window_summary.csv",
    "midpoints": RESULTS_DIR / "critical_midpoint_scaling.csv",
    "diagnostics": RESULTS_DIR / "critical_scaling_graph_diagnostics.csv",
}

print("results files:")
for p in sorted(RESULTS_DIR.glob("*")):
    print(" -", p)

missing = [str(path) for path in required_files.values() if not path.exists()]

if missing:
    print("\nMissing Notebook 16 outputs; compatible data will be regenerated internally.")
    for path in missing:
        print(" -", path)
else:
    print("\nAll Notebook 16 outputs found.")


In [ ]:

TOPOLOGY_PARAMS = {
    "ring_lattice": {
        "eta_inf": 0.135,
        "eta_shift": 0.060,
        "sigma_inf": 0.030,
        "sigma_scale": 0.080,
        "nu": 0.45,
        "beta": 0.40,
        "fragment_strength": 0.08,
        "modifier": 0.98,
    },
    "small_world": {
        "eta_inf": 0.160,
        "eta_shift": 0.075,
        "sigma_inf": 0.038,
        "sigma_scale": 0.095,
        "nu": 0.48,
        "beta": 0.37,
        "fragment_strength": 0.06,
        "modifier": 1.02,
    },
    "erdos_renyi": {
        "eta_inf": 0.120,
        "eta_shift": 0.055,
        "sigma_inf": 0.028,
        "sigma_scale": 0.075,
        "nu": 0.42,
        "beta": 0.45,
        "fragment_strength": 0.10,
        "modifier": 0.96,
    },
    "scale_free": {
        "eta_inf": 0.105,
        "eta_shift": 0.052,
        "sigma_inf": 0.025,
        "sigma_scale": 0.070,
        "nu": 0.44,
        "beta": 0.50,
        "fragment_strength": 0.14,
        "modifier": 0.93,
    },
    "modular_clustered": {
        "eta_inf": 0.092,
        "eta_shift": 0.048,
        "sigma_inf": 0.023,
        "sigma_scale": 0.065,
        "nu": 0.40,
        "beta": 0.52,
        "fragment_strength": 0.18,
        "modifier": 0.90,
    },
}

def simulate_cgcs_curve(topology, N, noise_grid, repeat=0):
    p = TOPOLOGY_PARAMS[topology]

    eta_c = finite_size_eta_c(p, N)
    sigma = finite_size_sigma(p, N)

    z = (noise_grid - eta_c) / sigma
    base = shared_profile(z)

    central_weight = np.exp(-0.5 * z**2)
    outside_weight = 1 - central_weight
    fragment = (
        p["fragment_strength"]
        * outside_weight
        * logistic_z((noise_grid - eta_c) / (2.0 * sigma))
    )

    rng = np.random.default_rng(
        40_000 + repeat + N + sum(ord(c) for c in topology)
    )
    noise_term = rng.normal(0, 0.010 * np.sqrt(32 / N), size=len(noise_grid))

    cgcs = p["modifier"] * base - fragment + noise_term
    return np.clip(cgcs, 0, 1)

def extract_transition_metrics(noise, cgcs):
    noise = np.asarray(noise, dtype=float)
    cgcs = np.asarray(cgcs, dtype=float)

    order = np.argsort(noise)
    noise = noise[order]
    cgcs = cgcs[order]

    midpoint_idx = int(np.argmin(np.abs(cgcs - MIDPOINT_LEVEL)))
    eta_mid = float(noise[midpoint_idx])

    deriv = np.gradient(cgcs, noise)
    max_slope_idx = int(np.argmax(np.abs(deriv)))
    eta_slope = float(noise[max_slope_idx])
    max_abs_slope = float(np.abs(deriv[max_slope_idx]))
    sigma_est = float(1 / max(4 * max_abs_slope, 1e-6))

    return {
        "eta_midpoint": eta_mid,
        "eta_max_slope": eta_slope,
        "max_abs_slope": max_abs_slope,
        "sigma_derivative": sigma_est,
    }

def regenerate_notebook16_compatible_data():
    metric_rows = []
    collapse_rows = []
    window_rows = []
    diag_rows = []

    repeats = 24

    for N in GRAPH_SIZES:
        for topology in TOPOLOGIES:
            all_curves = []
            for repeat in range(repeats):
                all_curves.append(simulate_cgcs_curve(topology, N, NOISE_GRID, repeat))

            mean_curve = np.array(all_curves).mean(axis=0)
            metrics = extract_transition_metrics(NOISE_GRID, mean_curve)

            metric_rows.append({
                "n_modules": N,
                "topology": topology,
                **metrics,
            })

            eta_mid = metrics["eta_midpoint"]
            sigma = max(metrics["sigma_derivative"], 1e-6)

            for eta, cgcs in zip(NOISE_GRID, mean_curve):
                z = (eta - eta_mid) / sigma
                sp = shared_profile(z)
                residual = float(cgcs - sp)

                collapse_rows.append({
                    "n_modules": N,
                    "topology": topology,
                    "link_noise": float(eta),
                    "cgcs": float(cgcs),
                    "z": float(z),
                    "shared_profile": float(sp),
                    "collapse_residual": residual,
                    "abs_residual": float(abs(residual)),
                })

            diag_rows.append({
                "n_modules": N,
                "topology": topology,
                "average_degree": np.nan,
                "degree_variance": np.nan,
                "clustering": np.nan,
                "path_length": np.nan,
                "largest_component_fraction": 1.0,
            })

    collapse_df = pd.DataFrame(collapse_rows)
    midpoint_df = pd.DataFrame(metric_rows)
    diag_df = pd.DataFrame(diag_rows)

    for (topology, N), sub in collapse_df.groupby(["topology", "n_modules"]):
        in_window = sub[sub["abs_residual"] <= EPSILON]
        if len(in_window) == 0:
            z_lower, z_upper, bandwidth, fraction = np.nan, np.nan, 0.0, 0.0
        else:
            z_lower = float(in_window["z"].min())
            z_upper = float(in_window["z"].max())
            bandwidth = float(z_upper - z_lower)
            fraction = float(len(in_window) / len(sub))

        window_rows.append({
            "n_modules": int(N),
            "topology": topology,
            "z_lower": z_lower,
            "z_upper": z_upper,
            "universality_bandwidth": bandwidth,
            "universality_fraction": fraction,
            "mean_abs_residual": float(sub["abs_residual"].mean()),
        })

    window_df = pd.DataFrame(window_rows)

    midpoint_df.to_csv(RESULTS_DIR / "critical_midpoint_scaling.csv", index=False)
    collapse_df.to_csv(RESULTS_DIR / "renormalized_collapse_data.csv", index=False)
    window_df.to_csv(RESULTS_DIR / "universality_window_summary.csv", index=False)
    diag_df.to_csv(RESULTS_DIR / "critical_scaling_graph_diagnostics.csv", index=False)

    return collapse_df, window_df, midpoint_df, diag_df


In [ ]:

if all(path.exists() for path in required_files.values()):
    collapse_df = pd.read_csv(required_files["collapse"])
    window_df = pd.read_csv(required_files["windows"])
    midpoint_df = pd.read_csv(required_files["midpoints"])
    diag_df = pd.read_csv(required_files["diagnostics"])
    data_source = "loaded existing Notebook 16 outputs"
else:
    collapse_df, window_df, midpoint_df, diag_df = regenerate_notebook16_compatible_data()
    data_source = "regenerated compatible Notebook 16 outputs internally"

collapse_df = collapse_df.replace([np.inf, -np.inf], np.nan).dropna(
    subset=["topology", "n_modules", "z", "cgcs"]
)

collapse_df["shared_profile_recomputed"] = shared_profile(collapse_df["z"].to_numpy())
collapse_df["residual"] = collapse_df["cgcs"] - collapse_df["shared_profile_recomputed"]
collapse_df["abs_residual"] = collapse_df["residual"].abs()
collapse_df["residual_energy"] = collapse_df["residual"] ** 2

collapse_df.to_csv(RESULTS_DIR / "residual_field_data.csv", index=False)

print("data source:", data_source)
print("collapse_df shape:", collapse_df.shape)
collapse_df.head()


## Smooth residual field helper

In [ ]:

def smooth_residual_on_grid(sub, z_grid, window=17, polyorder=3):
    ordered = sub.sort_values("z")
    tmp = (
        pd.DataFrame({
            "z": ordered["z"].to_numpy(dtype=float),
            "r": ordered["residual"].to_numpy(dtype=float),
        })
        .groupby("z", as_index=False)
        .mean()
    )

    z = tmp["z"].to_numpy()
    r = tmp["r"].to_numpy()

    if len(z) < 5:
        return np.full_like(z_grid, np.nan), np.full_like(z_grid, np.nan)

    r_grid = np.interp(z_grid, z, r, left=np.nan, right=np.nan)
    valid = np.isfinite(r_grid)

    if valid.sum() < window:
        return r_grid, r_grid

    r_valid = r_grid[valid]

    # Ensure odd window length and not longer than available valid samples.
    w = min(window, len(r_valid) if len(r_valid) % 2 == 1 else len(r_valid) - 1)
    w = max(w, polyorder + 2)
    if w % 2 == 0:
        w -= 1

    if w <= polyorder:
        return r_grid, r_grid

    smooth_valid = savgol_filter(r_valid, window_length=w, polyorder=polyorder, mode="interp")
    smooth_grid = r_grid.copy()
    smooth_grid[valid] = smooth_valid

    return r_grid, smooth_grid


## Figure 1 — Residual field by topology

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True, sharey=True)
axes = axes.ravel()

z_grid = np.linspace(-6, 6, 241)

for ax, topology in zip(axes, TOPOLOGIES):
    sub_topo = collapse_df[collapse_df["topology"] == topology]

    for N in GRAPH_SIZES:
        sub = sub_topo[sub_topo["n_modules"] == N]
        if len(sub) == 0:
            continue

        _, smooth_grid = smooth_residual_on_grid(sub, z_grid)

        ax.scatter(
            sub["z"],
            sub["residual"],
            s=14,
            alpha=0.22,
        )
        ax.plot(
            z_grid,
            smooth_grid,
            linewidth=2,
            label=f"N={N}"
        )

    ax.axhline(0, color="black", linewidth=1, linestyle="--")
    ax.set_title(topology.replace("_", " "))
    ax.set_xlabel("renormalized z")
    ax.set_ylabel("residual Δ(z)")
    ax.set_xlim(-6, 6)
    ax.grid(alpha=0.3)

axes[-1].axis("off")
axes[0].legend(fontsize=8)

plt.tight_layout()

fig_path = FIG_DIR / "residual_field_by_topology_v2.png"
plt.savefig(fig_path, dpi=220, bbox_inches="tight")
plt.show()

print(f"saved: {fig_path}")


## Residual localization

In [ ]:

localization_rows = []

for (topology, N), sub in collapse_df.groupby(["topology", "n_modules"]):
    energy = sub["residual_energy"].to_numpy()
    abs_res = sub["abs_residual"].to_numpy()

    total_energy = float(np.sum(energy))
    mean_abs = float(np.mean(abs_res))
    max_abs = float(np.max(abs_res))

    if len(energy) == 0 or total_energy <= 0:
        top_10pct_concentration = 0.0
    else:
        k = max(1, int(np.ceil(0.10 * len(energy))))
        top_energy = np.sort(energy)[-k:].sum()
        top_10pct_concentration = float(top_energy / total_energy)

    localization_rows.append({
        "topology": topology,
        "n_modules": int(N),
        "mean_abs_residual": mean_abs,
        "max_abs_residual": max_abs,
        "total_residual_energy": total_energy,
        "energy_concentration_top_10pct": top_10pct_concentration,
    })

localization_df = pd.DataFrame(localization_rows)
localization_df.to_csv(RESULTS_DIR / "residual_localization_summary.csv", index=False)

localization_df.head()


In [ ]:

plt.figure(figsize=(10, 6))

for topology in TOPOLOGIES:
    sub = localization_df[localization_df["topology"] == topology]
    plt.plot(
        sub["n_modules"],
        sub["energy_concentration_top_10pct"],
        marker="o",
        linewidth=2,
        label=topology.replace("_", " ")
    )

plt.xlabel("graph size N")
plt.ylabel("top 10% residual-energy concentration")
plt.title("Residual energy localization")
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.legend()

fig_path = FIG_DIR / "residual_energy_localization_v2.png"
plt.savefig(fig_path, dpi=220, bbox_inches="tight")
plt.show()


## Residual asymmetry

In [ ]:

symmetry_rows = []

for (topology, N), sub in collapse_df.groupby(["topology", "n_modules"]):
    left = sub[sub["z"] < 0]
    right = sub[sub["z"] >= 0]

    left_energy = float(left["residual_energy"].sum())
    right_energy = float(right["residual_energy"].sum())
    total_energy = left_energy + right_energy

    asymmetry = float((right_energy - left_energy) / total_energy) if total_energy > 0 else 0.0

    symmetry_rows.append({
        "topology": topology,
        "n_modules": int(N),
        "left_energy": left_energy,
        "right_energy": right_energy,
        "total_energy": total_energy,
        "asymmetry": asymmetry,
    })

symmetry_df = pd.DataFrame(symmetry_rows)
symmetry_df.to_csv(RESULTS_DIR / "residual_symmetry_summary.csv", index=False)

symmetry_df.head()


In [ ]:

plt.figure(figsize=(10, 6))

for topology in TOPOLOGIES:
    sub = symmetry_df[symmetry_df["topology"] == topology]
    plt.plot(
        sub["n_modules"],
        sub["asymmetry"],
        marker="o",
        linewidth=2,
        label=topology.replace("_", " ")
    )

plt.axhline(0, color="black", linestyle="--", linewidth=1)
plt.xlabel("graph size N")
plt.ylabel("residual asymmetry")
plt.title("Residual asymmetry by topology")
plt.grid(alpha=0.3)
plt.legend()

fig_path = FIG_DIR / "residual_asymmetry_by_topology_v2.png"
plt.savefig(fig_path, dpi=220, bbox_inches="tight")
plt.show()



## Smoothed residual bend energy

Raw second derivatives are noisy. This section uses Savitzky-Golay smoothing before estimating:

```text
d²Δ / dz²
```

We report this as **residual bend energy** rather than raw curvature.


In [ ]:

bend_rows = []
bend_profile_rows = []

z_grid = np.linspace(-6, 6, 241)

for (topology, N), sub in collapse_df.groupby(["topology", "n_modules"]):
    raw_grid, smooth_grid = smooth_residual_on_grid(sub, z_grid, window=17, polyorder=3)
    mask = np.isfinite(smooth_grid)

    if mask.sum() < 7:
        continue

    z_valid = z_grid[mask]
    r_smooth = smooth_grid[mask]

    first = np.gradient(r_smooth, z_valid)
    second = np.gradient(first, z_valid)

    # Bend energy: integrated square of smoothed second derivative.
    bend_energy = float(np.trapz(second ** 2, z_valid))
    mean_abs_bend = float(np.mean(np.abs(second)))
    max_abs_bend = float(np.max(np.abs(second)))

    bend_rows.append({
        "topology": topology,
        "n_modules": int(N),
        "mean_abs_bend": mean_abs_bend,
        "max_abs_bend": max_abs_bend,
        "bend_energy": bend_energy,
    })

    for z_val, bend_val in zip(z_valid, second):
        bend_profile_rows.append({
            "topology": topology,
            "n_modules": int(N),
            "z": float(z_val),
            "residual_bend": float(bend_val),
            "abs_residual_bend": float(abs(bend_val)),
        })

bend_df = pd.DataFrame(bend_rows)
bend_profile_df = pd.DataFrame(bend_profile_rows)

bend_df.to_csv(RESULTS_DIR / "residual_bend_energy_summary.csv", index=False)
bend_profile_df.to_csv(RESULTS_DIR / "residual_bend_profiles.csv", index=False)

bend_df.head()


In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True, sharey=True)
axes = axes.ravel()

for ax, topology in zip(axes, TOPOLOGIES):
    sub_topo = bend_profile_df[bend_profile_df["topology"] == topology]

    for N in GRAPH_SIZES:
        sub = sub_topo[sub_topo["n_modules"] == N]
        if len(sub) == 0:
            continue

        ax.plot(
            sub["z"],
            sub["residual_bend"],
            linewidth=1.8,
            label=f"N={N}"
        )

    ax.axhline(0, color="black", linestyle="--", linewidth=1)
    ax.set_title(topology.replace("_", " "))
    ax.set_xlabel("renormalized z")
    ax.set_ylabel("smoothed d²Δ/dz²")
    ax.set_xlim(-6, 6)
    ax.grid(alpha=0.3)

axes[-1].axis("off")
axes[0].legend(fontsize=8)

plt.tight_layout()

fig_path = FIG_DIR / "residual_bend_profiles_v2.png"
plt.savefig(fig_path, dpi=220, bbox_inches="tight")
plt.show()


In [ ]:

plt.figure(figsize=(10, 6))

for topology in TOPOLOGIES:
    sub = bend_df[bend_df["topology"] == topology]
    plt.plot(
        sub["n_modules"],
        sub["bend_energy"],
        marker="o",
        linewidth=2,
        label=topology.replace("_", " ")
    )

plt.xlabel("graph size N")
plt.ylabel("residual bend energy")
plt.title("Smoothed residual bend energy")
plt.grid(alpha=0.3)
plt.legend()

fig_path = FIG_DIR / "residual_bend_energy_by_topology_v2.png"
plt.savefig(fig_path, dpi=220, bbox_inches="tight")
plt.show()


## Residual entropy

In [ ]:

def normalized_entropy_from_energy(z, energy, bins=24):
    z = np.asarray(z)
    energy = np.asarray(energy)

    hist, _ = np.histogram(z, bins=bins, range=(-6, 6), weights=energy)
    total = hist.sum()

    if total <= 0:
        return 0.0

    p = hist / total
    p = p[p > 0]

    return float(-np.sum(p * np.log(p)) / np.log(bins))

entropy_rows = []

for (topology, N), sub in collapse_df.groupby(["topology", "n_modules"]):
    entropy_rows.append({
        "topology": topology,
        "n_modules": int(N),
        "residual_entropy": normalized_entropy_from_energy(
            sub["z"].to_numpy(),
            sub["residual_energy"].to_numpy(),
            bins=24
        ),
    })

entropy_df = pd.DataFrame(entropy_rows)
entropy_df.to_csv(RESULTS_DIR / "residual_entropy_summary.csv", index=False)

entropy_df.head()


In [ ]:

plt.figure(figsize=(10, 6))

for topology in TOPOLOGIES:
    sub = entropy_df[entropy_df["topology"] == topology]
    plt.plot(
        sub["n_modules"],
        sub["residual_entropy"],
        marker="o",
        linewidth=2,
        label=topology.replace("_", " ")
    )

plt.xlabel("graph size N")
plt.ylabel("normalized residual entropy")
plt.ylim(0, 1.05)
plt.title("Residual entropy by topology")
plt.grid(alpha=0.3)
plt.legend()

fig_path = FIG_DIR / "residual_entropy_by_topology_v2.png"
plt.savefig(fig_path, dpi=220, bbox_inches="tight")
plt.show()


## Residual spectral ratio — secondary diagnostic

In [ ]:

spectral_rows = []

z_fft = np.linspace(-6, 6, 256)

for (topology, N), sub in collapse_df.groupby(["topology", "n_modules"]):
    _, smooth_grid = smooth_residual_on_grid(sub, z_fft, window=17, polyorder=3)

    valid = np.isfinite(smooth_grid)
    if valid.sum() < 8:
        continue

    fill = np.nanmean(smooth_grid)
    r_grid = np.where(valid, smooth_grid, fill)

    r_centered = r_grid - np.mean(r_grid)
    fft_vals = np.fft.rfft(r_centered)
    mag = np.abs(fft_vals)
    power = mag ** 2

    non_dc_power = power.copy()
    non_dc_power[0] = 0

    total_power = float(non_dc_power.sum())

    if total_power <= 0:
        low_energy = 0.0
        high_energy = 0.0
        spectral_ratio = 0.0
        dominant_mode = 0
    else:
        cutoff = max(2, int(0.20 * len(non_dc_power)))
        low_energy = float(non_dc_power[1:cutoff].sum() / total_power)
        high_energy = float(non_dc_power[cutoff:].sum() / total_power)
        spectral_ratio = float(high_energy / max(low_energy, 1e-9))
        dominant_mode = int(np.argmax(non_dc_power))

    spectral_rows.append({
        "topology": topology,
        "n_modules": int(N),
        "low_frequency_energy": low_energy,
        "high_frequency_energy": high_energy,
        "spectral_ratio": spectral_ratio,
        "dominant_mode": dominant_mode,
        "total_spectral_power": total_power,
    })

spectral_df = pd.DataFrame(spectral_rows)
spectral_df.to_csv(RESULTS_DIR / "residual_spectral_summary.csv", index=False)

spectral_df.head()


In [ ]:

plt.figure(figsize=(10, 6))

for topology in TOPOLOGIES:
    sub = spectral_df[spectral_df["topology"] == topology]
    plt.plot(
        sub["n_modules"],
        sub["spectral_ratio"],
        marker="o",
        linewidth=2,
        label=topology.replace("_", " ")
    )

plt.xlabel("graph size N")
plt.ylabel("high / low frequency residual energy")
plt.title("Residual spectral ratio by topology")
plt.grid(alpha=0.3)
plt.legend()

fig_path = FIG_DIR / "residual_spectral_ratio_by_topology_v2.png"
plt.savefig(fig_path, dpi=220, bbox_inches="tight")
plt.show()


## Combined residual geometry table

In [ ]:

geometry_df = (
    localization_df
    .merge(symmetry_df[["topology", "n_modules", "asymmetry"]], on=["topology", "n_modules"])
    .merge(entropy_df, on=["topology", "n_modules"])
    .merge(spectral_df[["topology", "n_modules", "spectral_ratio"]], on=["topology", "n_modules"])
    .merge(bend_df[["topology", "n_modules", "bend_energy", "mean_abs_bend"]], on=["topology", "n_modules"])
)

geometry_df.to_csv(RESULTS_DIR / "residual_geometry_features.csv", index=False)
geometry_df.head()


## Hero figure — residual geometry map

In [ ]:

plt.figure(figsize=(10, 7))

max_bend = max(geometry_df["bend_energy"].max(), 1e-9)

for topology in TOPOLOGIES:
    sub = geometry_df[geometry_df["topology"] == topology]

    sizes = 80 + 500 * (sub["bend_energy"] / max_bend)

    plt.scatter(
        sub["residual_entropy"],
        sub["asymmetry"],
        s=sizes,
        alpha=0.75,
        label=topology.replace("_", " ")
    )

    for _, row in sub.iterrows():
        plt.annotate(
            f"N={int(row['n_modules'])}",
            xy=(row["residual_entropy"], row["asymmetry"]),
            xytext=(4, 4),
            textcoords="offset points",
            fontsize=7,
        )

plt.axhline(0, color="black", linestyle="--", linewidth=1)
plt.xlabel("residual entropy")
plt.ylabel("residual asymmetry")
plt.title("Residual topology geometry\n(size = smoothed residual bend energy)")
plt.grid(alpha=0.3)
plt.legend(fontsize=8)

fig_path = FIG_DIR / "residual_geometry_map_v2.png"
plt.savefig(fig_path, dpi=240, bbox_inches="tight")
plt.show()


## Compact residual dashboard

In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True)
axes = axes.ravel()

metrics = [
    ("mean_abs_residual", "mean |residual|"),
    ("residual_entropy", "residual entropy"),
    ("asymmetry", "asymmetry"),
    ("bend_energy", "residual bend energy"),
]

for ax, (metric, ylabel) in zip(axes, metrics):
    for topology in TOPOLOGIES:
        sub = geometry_df[geometry_df["topology"] == topology]
        ax.plot(
            sub["n_modules"],
            sub[metric],
            marker="o",
            linewidth=2,
            label=topology.replace("_", " ")
        )

    ax.set_title(ylabel)
    ax.set_xlabel("graph size N")
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)

axes[0].legend(fontsize=8)

plt.tight_layout()

fig_path = FIG_DIR / "residual_geometry_dashboard_v2.png"
plt.savefig(fig_path, dpi=220, bbox_inches="tight")
plt.show()


## Summary export

In [ ]:

summary_by_topology = []

for topology in TOPOLOGIES:
    sub = geometry_df[geometry_df["topology"] == topology]
    summary_by_topology.append({
        "topology": topology,
        "mean_abs_residual": float(sub["mean_abs_residual"].mean()),
        "mean_entropy": float(sub["residual_entropy"].mean()),
        "mean_asymmetry": float(sub["asymmetry"].mean()),
        "mean_spectral_ratio": float(sub["spectral_ratio"].mean()),
        "mean_bend_energy": float(sub["bend_energy"].mean()),
    })

summary_topology_df = pd.DataFrame(summary_by_topology)
summary_topology_df.to_csv(RESULTS_DIR / "residual_topology_summary_by_topology.csv", index=False)

most_localized = localization_df.sort_values("energy_concentration_top_10pct", ascending=False).iloc[0]
highest_entropy = entropy_df.sort_values("residual_entropy", ascending=False).iloc[0]
highest_bend = bend_df.sort_values("bend_energy", ascending=False).iloc[0]

summary = {
    "notebook": "17_residual_topology_geometry_v2.ipynb",
    "data_source": data_source,
    "core_claim": (
        "Residuals around the shared bounded transition manifold are structured, "
        "topology-specific, and measurable through localization, asymmetry, "
        "entropy, spectral diagnostics, and smoothed residual bend energy."
    ),
    "interpretation": (
        "The shared transition persists; topology reappears as structured "
        "residual geometry."
    ),
    "most_localized_residual": {
        "topology": most_localized["topology"],
        "n_modules": int(most_localized["n_modules"]),
        "energy_concentration_top_10pct": float(most_localized["energy_concentration_top_10pct"]),
    },
    "highest_residual_entropy": {
        "topology": highest_entropy["topology"],
        "n_modules": int(highest_entropy["n_modules"]),
        "residual_entropy": float(highest_entropy["residual_entropy"]),
    },
    "highest_bend_energy": {
        "topology": highest_bend["topology"],
        "n_modules": int(highest_bend["n_modules"]),
        "bend_energy": float(highest_bend["bend_energy"]),
    },
    "paper_figures_recommended": [
        "residual_field_by_topology_v2.png",
        "residual_geometry_map_v2.png",
        "residual_energy_localization_v2.png",
        "residual_asymmetry_by_topology_v2.png",
        "residual_entropy_by_topology_v2.png"
    ],
    "secondary_figures": [
        "residual_bend_profiles_v2.png",
        "residual_bend_energy_by_topology_v2.png",
        "residual_spectral_ratio_by_topology_v2.png",
        "residual_geometry_dashboard_v2.png"
    ],
    "results": [
        "residual_field_data.csv",
        "residual_localization_summary.csv",
        "residual_symmetry_summary.csv",
        "residual_bend_energy_summary.csv",
        "residual_bend_profiles.csv",
        "residual_entropy_summary.csv",
        "residual_spectral_summary.csv",
        "residual_geometry_features.csv",
        "residual_topology_summary_by_topology.csv",
        "residual_topology_geometry_summary.json",
    ],
}

summary_path = RESULTS_DIR / "residual_topology_geometry_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

doc_lines = [
    "# Notebook 17 v2 — Residual Topology Geometry",
    "",
    "**Core claim:** residuals around the shared bounded transition manifold are structured, topology-specific, and measurable.",
    "",
    "> The shared transition persists; topology reappears as structured residual geometry.",
    "",
    f"Data source: `{data_source}`",
    "",
    "Recommended paper figures:",
    "",
    "- `figures/residual_field_by_topology_v2.png`",
    "- `figures/residual_geometry_map_v2.png`",
    "- `figures/residual_energy_localization_v2.png`",
    "- `figures/residual_asymmetry_by_topology_v2.png`",
    "- `figures/residual_entropy_by_topology_v2.png`",
    "",
]

doc_path = DOCS_DIR / "notebook_17_residual_topology_geometry_v2.md"
doc_path.write_text("\n".join(doc_lines), encoding="utf-8")

print(json.dumps(summary, indent=2))
print(f"saved: {summary_path}")
print(f"saved: {doc_path}")



## Final interpretation

Careful conclusion:

```text
Collapse removes the dominant transition profile but does not erase topology.
Residual fields remain structured and topology-specific, showing organized
fragmentation around the shared bounded manifold.
```

Short version:

```text
The shared transition persists; topology reappears as structured residual geometry.
```


## Optional export zip

In [ ]:

zip_path = Path("notebook_17_v2_outputs.zip")

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in [FIG_DIR, RESULTS_DIR, DOCS_DIR]:
        if folder.exists():
            for file in folder.glob("*"):
                if file.is_file():
                    zf.write(file)

print(f"Created: {zip_path}")

# Optional Colab download:
# from google.colab import files
# files.download(str(zip_path))
